## SacreBLEU's chrF implementation


| parameter  | default | role                                     |
|------------|---------|------------------------------------------|
| char_order | 6       | Maximum character n-gram order           |
| word_order | 2       | Maximum word n-gram order                |
| beta       | 2       | Recall weight                            |
| whitespace | False   | Whether spaces are treated as characters |


* word_order = 2 

    include word 1-grams and word 2-grams in addition to character n-grams, which is the chrF++ metric that [Popović](https://github.com/m-popovic/chrF) proposed

    c.f. the arithmetic mean is used for n-gram averaging

* word_order = 0 

    compute only character n-gram F-score, which is the original chrF metric proposed by [Popović (2015)](https://aclanthology.org/W15-3049/)

* whitespace = False
  
    whitespaces are all excluded before extracting character n-grams

* whitespace = True
  
    whitespaces are treated as characters before extracting character n-grams 

Word n-grams capture lexical correctness and word order more explicitly while character n-grams focus on character-level orthographic and spacing differences.

If we want spacing-convention differences between NK and SK to actually count toward chrF3, use whitespace=True

In [1]:
from sacrebleu.metrics import CHRF
chrf = CHRF(word_order=0, beta=3)
print(chrf.sentence_score("책상밑으로", ["책상 밑으로"]).score)

100.0


In [2]:
from sacrebleu.metrics import CHRF
chrf = CHRF(word_order=0, beta=3, whitespace=True)
print(chrf.sentence_score("책상밑으로", ["책상 밑으로"]).score)

34.32572050027189


Q. Could it be that chrF score is penalized a lot because "책상 밑으로" is a short sequence?

Yes. The space is only one character, but affects many available higher-order n-grams. The key point is "What proportion of the hypothesis and reference n-grams match at each order?"

Inserting a space (책상밑으로 -> 책상 밑으로) changes 

2-gram \
: 책상 / 상밑 / 밑으 / 으로 into 책상 / 상_ / _밑 / 밑으 / 으로

3-gram \
: 책상밑 / 상밑으 / 밑으로 into 책상_ / 상_밑 / _밑으 / 밑으로

4-gram \
: 책상밑으 / 상밑으로 into 책상_밑 / 상_밑으 / _밑으로

5-gram \
: 책상밑으로 into 책상_밑으 / 상_밑으로

A sentence of length $L$ contains $L-n+1$ $n$-grams of order $n$.

For a 45-character sentence, that is roughly:

* 45 unigrams
* 44 bigrams
* 43 trigrams
* 42 four-grams
* 41 five-grams
* 40 six-grams

By contrast, in a 5-chracter sequence such as 책상밑으로, there are only:

* 5 unigrams
* 4 bigrams
* 3 trigrams
* 2 four-grams
* 1 five-gram

In a longer sentence, a single spacing difference affects only a small proportion of the available character n-grams. In a short sentence, however, the same difference disrupts a much larger proportion of the available n-grams, especially at higher orders.

In [3]:
from sacrebleu.metrics import CHRF
chrf = CHRF(word_order=0, beta=3)
print(chrf.sentence_score("나는 걸상을 뛰어넘기도 하고 책상밑으로 기어나가기도 하면서 난로 옆까지 갔다.", ["나는 걸상을 뛰어넘기도 하고 책상 밑으로 기어나가기도 하면서 난로 옆까지 갔다."]).score)

100.0


In [4]:
from sacrebleu.metrics import CHRF
chrf = CHRF(word_order=0, beta=3, whitespace=True)
print(chrf.sentence_score("나는 걸상을 뛰어넘기도 하고 책상밑으로 기어나가기도 하면서 난로 옆까지 갔다.", ["나는 걸상을 뛰어넘기도 하고 책상 밑으로 기어나가기도 하면서 난로 옆까지 갔다."]).score)

91.60275692275921
